In [1]:
# Cek GPU
import torch
print(torch.cuda.is_available())

True


In [2]:
#!pip install transformers datasets accelerate
!pip install transformers accelerate datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=0dec54a2e0c476875c66246c9b75ddb63e190f98620a066448e5c98712d375a5
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [3]:
# Import
import transformers
import accelerate

from datasets import Dataset
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
# from datasets import load_metric # Sudah Tidak Digunakan
import evaluate
import json
import os

In [4]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**DATASET**

In [5]:
!tar -xzf "/content/drive/MyDrive/Colab Notebooks/Data/liputan6_data.tar.gz" -C "/content/sample_data"

In [6]:
# Load Dataset
train_path = "/content/sample_data/liputan6_data/canonical/train"
dev_path = "/content/sample_data/liputan6_data/canonical/dev"
test_path = "/content/sample_data/liputan6_data/canonical/test"


def load_all_json(folder):
    data = []

    files = sorted(os.listdir(folder))[:10]

    for filename in files:
        if filename.endswith(".json"):
            file_path = os.path.join(folder, filename)

            with open(file_path, 'r') as f:
                item = json.load(f)
                data.append(item)

    return data

train_data = load_all_json(train_path)
dev_data = load_all_json(dev_path)
test_data = load_all_json(test_path)

print("Jumlah data train_data:", len(train_data))
print("Contoh data train_data:", train_data[0])
print("--"*50)
print("Jumlah data dev_data:", len(dev_data))
print("Contoh data dev_data:", dev_data[0])
print("--"*50)
print("Jumlah data test_data:", len(test_data))
print("Contoh data test_data:", test_data[0])

Jumlah data train_data: 10
Contoh data train_data: {'id': 100000, 'url': 'https://www.liputan6.com/news/read/100000/yudhoyono-berharap-masalah-kemiskinan-menjadi-bahasan-penting', 'clean_article': [['Liputan6', '.', 'com', ',', 'Jakarta', ':', 'Presiden', 'Susilo', 'Bambang', 'Yudhoyono', 'menekankan', 'bahwa', 'tantangan', 'terbesar', 'yang', 'dihadapi', 'bangsa-bangsa', 'Asia', 'dan', 'Afrika', 'saat', 'ini', 'adalah', 'masalah', 'kemiskinan', 'yang', 'sangat', 'buruk', '.'], ['Yudhoyono', 'berharap', 'masalah', 'ini', 'menjadi', 'pembahasan', 'penting', 'dalam', 'Konferensi', 'Tingkat', 'Tinggi', 'Asia-Afrika', '.'], ['Demikian', 'pidato', 'Yudhoyono', 'saat', 'membuka', 'KTT', 'Asia-Afrika', 'di', 'Jakarta', 'Convention', 'Centre', ',', 'Jakarta', ',', 'Jumat', '(', '22/4', ')', '[', 'baca', ':', 'Presiden', 'Yudhoyono', 'Resmi', 'Membuka', 'KAA', ']', '.'], ['Pada', 'awal', 'pidatonya', ',', 'Yudhoyono', 'para', 'peserta', 'untuk', 'mengheningkan', 'cipta', 'sejenak', 'bagi', 'kor

In [7]:
# Convert Dataset Untuk mengambil clean_artikel dan clean_summary saja

def convert_to_text(data):
    new_data = []

    for item in data:
        # Gabungkan artikel
        article = " ".join(
            [" ".join(sentence) for sentence in item["clean_article"]]
        )

        # Gabungkan summary
        summary = " ".join(
            [" ".join(sentence) for sentence in item["clean_summary"]]
        )

        new_data.append({
            "clean_article": article,
            "clean_summary": summary
        })

    return new_data

train_data_convert = convert_to_text(train_data)
dev_data_convert = convert_to_text(dev_data)
test_data_convert = convert_to_text(test_data)

In [8]:
# Convert ke HuggingFace Dataset

train_dataset = Dataset.from_list(train_data_convert)
dev_dataset   = Dataset.from_list(dev_data_convert)
test_dataset  = Dataset.from_list(test_data_convert)

In [9]:
# Load Model & Tokenizer

model_name = "facebook/bart-base"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [10]:
# Tokenisasi

def tokenize_function(example):
    inputs = tokenizer(
        example["clean_article"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["clean_summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels_ids = labels["input_ids"]

    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels_ids
    ]

    inputs["labels"] = labels_ids
    return inputs

In [11]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_dev   = dev_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [12]:
# Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [13]:
# ROUGE Metric
import evaluate

rouge = evaluate.load("rouge")



In [27]:
import evaluate
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = [
        [(l if l != -100 else tokenizer.pad_token_id) for l in label]
        for label in labels
    ]

    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

In [22]:
# Training Proccess

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    #evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    fp16=True
    #,predict_with_generate=True
)

In [29]:
# Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [31]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=2.6208767890930176, metrics={'train_runtime': 51.1078, 'train_samples_per_second': 0.196, 'train_steps_per_second': 0.059, 'total_flos': 3048682291200.0, 'train_loss': 2.6208767890930176, 'epoch': 1.0})

In [32]:
# save model
trainer.save_model("/content/sample_data/Model")
tokenizer.save_pretrained("/content/sample_data/Model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/sample_data/Model/tokenizer_config.json',
 '/content/sample_data/Model/tokenizer.json')

In [33]:
# Test - pakai test, bukan dev

results = trainer.evaluate(tokenized_test)
print(results)

TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

In [ ]:
# Generate Summary Manual

def generate_summary(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True).to("cuda")

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=100,
        num_beams=4
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


In [ ]:
# Coba 5 Data
for i in range(5):
    text = test_convert[i]["clean_article"]
    ref  = test_convert[i]["clean_summary"]

    pred = generate_summary(text)

    print("PRED:", pred)
    print("REF :", ref)
    print("-"*50)